# NB08 — Pfam & COG CWM Metal Associations

Runs the same L0–L6 vectorized FWL as NB02 (KO CWM × metals) for two
complementary annotation systems:

- **Pfam domains** (from `eggnog_mapper_annotations.PFAMs`, pipe-separated)
- **COG functional categories** (from `eggnog_mapper_annotations.COG_category`, letter codes)

Both use the same ke_pangenome genus-level prevalence approach as NB01.
Results are compared per-metal to the KO FDR hits from NB02.

**Strategy:** two long Spark queries (Pfam ~30–90 min, COG ~20–60 min) saved to
parquet; all FWL runs locally in vectorized numpy.


In [1]:
import sys, os
sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, PALETTE, METAL_COLORS, FIGW, ROW_H, grid_h, annotate_n
apply_style()

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import t as t_dist
from scipy.stats import entropy as scipy_entropy
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

PROJECT = Path('/home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_cwm')
DATA    = PROJECT / 'data'
FIGS    = PROJECT / 'figures'

try:
    from berdl_notebook_utils.setup_spark_session import get_spark_session
    spark = get_spark_session()
except Exception as e:
    print(f'berdl_notebook_utils failed ({e}), trying getOrCreate')
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
print(f'Spark {spark.version}')
print(f'DATA: {DATA}')


Spark 4.0.1
DATA: /home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_cwm/data


In [2]:
# Load all NB00/NB01/NB02 parquets needed to reconstruct the covariate matrix.
thinned      = pd.read_parquet(DATA / 'nb00_thinned_samples.parquet')
genus_counts = pd.read_parquet(DATA / 'nb01_genus_counts.parquet')
soil_props   = pd.read_parquet(DATA / 'nb02_soil_props.parquet')
lith_raw     = pd.read_parquet(DATA / 'nb02_lithology.parquet')
wc           = pd.read_parquet(DATA / 'nb02_worldclim_seasonal.parquet')
mine_dist    = pd.read_parquet(DATA / 'nb02_mine_dist.parquet')
genus_phylum = pd.read_parquet(DATA / 'nb02_genus_phylum.parquet')
combined     = pd.read_parquet(DATA / 'nb02_combined_metals.parquet')

# ph_final: measured pH > OLM÷10 > SoilGrids raster (covers 966/1006 otherwise-missing)
ph_raw = pd.to_numeric(thinned['ph'], errors='coerce')
ph_olm = pd.to_numeric(thinned['olm_soil_ph_0cm_H2O'], errors='coerce') / 10.0
thinned['ph_final'] = ph_raw.where(ph_raw.notna(),
    ph_olm.where(ph_olm.notna(), thinned['ph_soilgrids']))
thinned['ph_is_modelled'] = thinned['ph'].isna().astype(np.float32)

# Lithology dummies
lith_dummies = pd.get_dummies(lith_raw.set_index('sample_id')['lith_class'],
                               prefix='lith').astype(np.float32)

# log_mine_km (mine_dist parquet only has mine_dist_km — compute here)
mine_dist['log_mine_km'] = np.log10(mine_dist['mine_dist_km'].clip(lower=0.1))

# Genus RA for Shannon + phylum breakdown
total_per_sample = genus_counts.groupby('sample_id')['genus_count'].sum().rename('total')
gc2 = genus_counts.join(total_per_sample, on='sample_id')
gc2['genus_ra'] = gc2['genus_count'] / gc2['total']

shannon_df = (gc2.groupby('sample_id')['genus_ra']
              .apply(lambda x: scipy_entropy(x[x > 0]))
              .rename('shannon').reset_index())

phy_map = genus_phylum.set_index('genus_lower')['phylum_lower'].to_dict()
gc2['phylum'] = gc2['genus_lower'].map(phy_map)
top8_phyla = (gc2.groupby('phylum')['genus_ra'].mean()
              .sort_values(ascending=False).dropna().head(8).index.tolist())
phy_ra = (gc2[gc2['phylum'].isin(top8_phyla)]
          .groupby(['sample_id', 'phylum'])['genus_ra'].sum()
          .unstack(fill_value=0.0))
phy_wide = phy_ra.reindex(columns=top8_phyla, fill_value=0.0)

print(f'Thinned: {len(thinned):,}  genus_counts: {len(genus_counts):,}')
print(f'top8_phyla: {top8_phyla}')


Thinned: 4,884  genus_counts: 844,984
top8_phyla: ['bryopsida', 'wallemiomycetes', 'insecta', 'sordariomycetes', 'nitrososphaeria', 'fibrobacteria', 'blastocladiomycetes', 'elardia']


In [3]:
# Assemble per-sample covariate matrix (mirrors NB02 exactly).
base = thinned[['sample_id', 'lat', 'lon', 'ph_final', 'ph_is_modelled',
                 'dem_elevation_m', 'era5_mean_2m_air_temp_k',
                 'era5_total_precipitation_mm', 'ndvi']].copy().set_index('sample_id')
base = base.join(soil_props.set_index('sample_id'), how='left')
base = base.join(lith_dummies, how='left')
base = base.join(wc.set_index('sample_id'), how='left')
base = base.join(mine_dist[['sample_id', 'log_mine_km']].set_index('sample_id'), how='left')
base = base.join(shannon_df.set_index('sample_id'), how='left')
base = base.join(phy_wide, how='left')
region_dummies = pd.get_dummies(combined.set_index('sample_id')['region'],
                                 prefix='region', drop_first=True).astype(np.float32)
base = base.join(region_dummies, how='left')

# pH cubic polynomial
ph_arr  = base['ph_final'].values.astype(np.float64)
ph_mean = np.nanmean(ph_arr)
ph_c    = np.where(np.isnan(ph_arr), 0.0, ph_arr - ph_mean)
ph_basis = np.column_stack([ph_c, ph_c**2, ph_c**3])

def impute_median(arr):
    arr = arr.copy()
    mask = np.isnan(arr)
    if mask.any():
        arr[mask] = np.nanmedian(arr)
    return arr

lith_cols   = [c for c in base.columns if c.startswith('lith_')]
region_cols = [c for c in base.columns if c.startswith('region_')]

Z_blocks = {
    'L1': ph_basis,
    'L2': np.column_stack([
        impute_median(base['clay_pct'].values.astype(np.float64)),
        impute_median(base['som_pct'].values.astype(np.float64)),
        impute_median(base['bulk_density'].values.astype(np.float64)),
        *[base[c].fillna(0).values.astype(np.float64) for c in lith_cols],
    ]),
    'L3': impute_median(base['log_mine_km'].values.astype(np.float64)).reshape(-1, 1),
    'L4': np.column_stack([
        impute_median(base['era5_mean_2m_air_temp_k'].values.astype(np.float64)),
        impute_median(base['era5_total_precipitation_mm'].values.astype(np.float64)),
        impute_median(base['temp_seasonality'].values.astype(np.float64)),
        impute_median(base['precip_seasonality'].values.astype(np.float64)),
    ]),
    'L5': np.column_stack([
        impute_median(base['shannon'].values.astype(np.float64)),
        *[base[p].fillna(0).values.astype(np.float64) for p in top8_phyla if p in base.columns],
    ]),
    'L6': np.column_stack([
        impute_median(base['dem_elevation_m'].values.astype(np.float64)),
        base['ndvi'].fillna(0).values.astype(np.float64),
    ]),
}

def build_Z(level_int):
    keys = ['L1', 'L2', 'L3', 'L4', 'L5', 'L6'][:level_int]
    parts = [np.ones((len(base), 1))]
    parts += [Z_blocks[k] for k in keys]
    if region_cols:
        parts.append(np.column_stack([base[c].fillna(0).values for c in region_cols]))
    return np.column_stack(parts).astype(np.float64)

print('Covariate matrix shapes:')
for lv in range(7):
    Z = build_Z(lv)
    n_finite = np.all(np.isfinite(Z), axis=1).sum()
    print(f'  L{lv}: {Z.shape}, rows fully finite: {n_finite:,}')


Covariate matrix shapes:
  L0: (4884, 3), rows fully finite: 4,884
  L1: (4884, 6), rows fully finite: 4,884
  L2: (4884, 22), rows fully finite: 4,884
  L3: (4884, 23), rows fully finite: 4,884
  L4: (4884, 27), rows fully finite: 4,884
  L5: (4884, 36), rows fully finite: 4,884
  L6: (4884, 38), rows fully finite: 4,884


In [4]:
# Spark: Pfam domain prevalence per genus from ke_pangenome.
# Prevalence = fraction of species clades in genus g carrying at least one gene cluster
# annotated with the Pfam domain (eggnog_mapper_annotations.PFAMs, pipe-separated).
# Expected runtime: 30–90 min.

if (DATA / 'nb08_pfam_prevalence.parquet').exists():
    print('nb08_pfam_prevalence.parquet exists — loading from disk')
    pfam_prev = pd.read_parquet(DATA / 'nb08_pfam_prevalence.parquet')
else:
    sql_pfam = """
        WITH genus_denominator AS (
            SELECT LOWER(REGEXP_EXTRACT(GTDB_taxonomy, 'g__([^;]+)', 1)) AS genus_lower,
                   CAST(COUNT(*) AS DOUBLE) AS n_total
            FROM   kbase.ke_pangenome.gtdb_species_clade
            WHERE  GTDB_taxonomy LIKE '%g__%'
            GROUP BY LOWER(REGEXP_EXTRACT(GTDB_taxonomy, 'g__([^;]+)', 1))
        ),
        pfam_exploded AS (
            SELECT e.query_name AS gene_cluster_id, pfam_id
            FROM   kbase.ke_pangenome.eggnog_mapper_annotations e
            LATERAL VIEW explode(split(e.PFAMs, ',')) pfam_table AS pfam_id
            WHERE  e.PFAMs IS NOT NULL AND e.PFAMs != '' AND e.PFAMs != '-'
              AND  pfam_id IS NOT NULL AND pfam_id != ''
        )
        SELECT LOWER(REGEXP_EXTRACT(sc.GTDB_taxonomy, 'g__([^;]+)', 1)) AS genus_lower,
               pe.pfam_id,
               CAST(COUNT(DISTINCT gc.gtdb_species_clade_id) AS DOUBLE)
                   / den.n_total AS prevalence
        FROM   pfam_exploded pe
        JOIN   kbase.ke_pangenome.gene_cluster   gc  ON pe.gene_cluster_id = gc.gene_cluster_id
        JOIN   kbase.ke_pangenome.gtdb_species_clade sc
               ON gc.gtdb_species_clade_id = sc.gtdb_species_clade_id
        JOIN   genus_denominator den
               ON LOWER(REGEXP_EXTRACT(sc.GTDB_taxonomy, 'g__([^;]+)', 1)) = den.genus_lower
        WHERE  sc.GTDB_taxonomy LIKE '%g__%'
        GROUP BY LOWER(REGEXP_EXTRACT(sc.GTDB_taxonomy, 'g__([^;]+)', 1)),
                 pe.pfam_id, den.n_total
    """
    print('Running Pfam prevalence Spark query (expect 30-90 min)...')
    result = spark.sql(sql_pfam)
    result.attrs = {}
    pfam_prev = result.toPandas()
    pfam_prev.attrs = {}
    pfam_prev.to_parquet(DATA / 'nb08_pfam_prevalence.parquet', index=False)
    print(f'Saved {len(pfam_prev):,} genus×Pfam rows')

print(f'\npfam_prev: {len(pfam_prev):,} rows')
print(f'Unique genera: {pfam_prev["genus_lower"].nunique():,}')
print(f'Unique Pfams:  {pfam_prev["pfam_id"].nunique():,}')
print(pfam_prev.head(3).to_string(index=False))


nb08_pfam_prevalence.parquet exists — loading from disk



pfam_prev: 15,104,947 rows
Unique genera: 8,419


Unique Pfams:  10,954
    genus_lower        pfam_id  prevalence
   pelagibacter NAD_Gly3P_dh_C    0.978142
2-12-full-35-15 DNA_pol_A_exo1    1.000000
       caitza01         DUF218    1.000000


In [5]:
# Spark: COG functional category prevalence per genus from ke_pangenome.
# COG_category is a string like 'CK' — each letter is a separate category.
# Expected runtime: 20–60 min.

if (DATA / 'nb08_cog_prevalence.parquet').exists():
    print('nb08_cog_prevalence.parquet exists — loading from disk')
    cog_prev = pd.read_parquet(DATA / 'nb08_cog_prevalence.parquet')
else:
    sql_cog = """
        WITH genus_denominator AS (
            SELECT LOWER(REGEXP_EXTRACT(GTDB_taxonomy, 'g__([^;]+)', 1)) AS genus_lower,
                   CAST(COUNT(*) AS DOUBLE) AS n_total
            FROM   kbase.ke_pangenome.gtdb_species_clade
            WHERE  GTDB_taxonomy LIKE '%g__%'
            GROUP BY LOWER(REGEXP_EXTRACT(GTDB_taxonomy, 'g__([^;]+)', 1))
        ),
        cog_exploded AS (
            SELECT e.query_name AS gene_cluster_id, cog_letter
            FROM   kbase.ke_pangenome.eggnog_mapper_annotations e
            LATERAL VIEW explode(split(e.COG_category, '')) cog_table AS cog_letter
            WHERE  e.COG_category IS NOT NULL AND e.COG_category != '' AND e.COG_category != '-'
              AND  cog_letter IS NOT NULL AND LENGTH(cog_letter) = 1
              AND  cog_letter RLIKE '[A-Z]'
        )
        SELECT LOWER(REGEXP_EXTRACT(sc.GTDB_taxonomy, 'g__([^;]+)', 1)) AS genus_lower,
               ce.cog_letter,
               CAST(COUNT(DISTINCT gc.gtdb_species_clade_id) AS DOUBLE)
                   / den.n_total AS prevalence
        FROM   cog_exploded ce
        JOIN   kbase.ke_pangenome.gene_cluster   gc  ON ce.gene_cluster_id = gc.gene_cluster_id
        JOIN   kbase.ke_pangenome.gtdb_species_clade sc
               ON gc.gtdb_species_clade_id = sc.gtdb_species_clade_id
        JOIN   genus_denominator den
               ON LOWER(REGEXP_EXTRACT(sc.GTDB_taxonomy, 'g__([^;]+)', 1)) = den.genus_lower
        WHERE  sc.GTDB_taxonomy LIKE '%g__%'
        GROUP BY LOWER(REGEXP_EXTRACT(sc.GTDB_taxonomy, 'g__([^;]+)', 1)),
                 ce.cog_letter, den.n_total
    """
    print('Running COG prevalence Spark query (expect 20-60 min)...')
    result = spark.sql(sql_cog)
    result.attrs = {}
    cog_prev = result.toPandas()
    cog_prev.attrs = {}
    cog_prev.to_parquet(DATA / 'nb08_cog_prevalence.parquet', index=False)
    print(f'Saved {len(cog_prev):,} genus×COG rows')

print(f'\ncog_prev: {len(cog_prev):,} rows')
print(f'Unique genera: {cog_prev["genus_lower"].nunique():,}')
print(f'Unique COG letters: {sorted(cog_prev["cog_letter"].unique())}')
print(cog_prev.head(3).to_string(index=False))


nb08_cog_prevalence.parquet exists — loading from disk

cog_prev: 175,861 rows
Unique genera: 8,419
Unique COG letters: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'S', 'T', 'U', 'V', 'W', 'Y', 'Z']
genus_lower cog_letter  prevalence
     sxxk01          U         1.0
     dusc01          G         1.0
     cag-95          E         1.0


In [6]:
# Compute Pfam CWM: CWM[s, pfam] = Σ_genus(genus_RA[s,genus] × pfam_prevalence[genus,pfam]).
# Filter Pfams present in ≥30 samples (same threshold as NB02 KO filter).

PFAM_MIN_GENERA = 5   # Pfam must appear in ≥5 genera
PFAM_MIN_SAMPLES = 30 # CWM>0 in ≥30 samples

# Wide genus×Pfam prevalence matrix
pfam_wide_full = pfam_prev.pivot_table(
    index='genus_lower', columns='pfam_id', values='prevalence', fill_value=0.0)

# Top-N filter to avoid OOM: keep 5000 most prevalent Pfam domains
TOP_PFAM = 5000
pfam_mean_prev = pfam_wide_full.mean(axis=0).sort_values(ascending=False)
pfam_keep_topn = pfam_mean_prev.head(TOP_PFAM).index
pfam_wide_full = pfam_wide_full[pfam_keep_topn]
print(f'Pfam domains after top-{TOP_PFAM} prevalence filter: {pfam_wide_full.shape[1]:,}')

# Filter Pfams by genus count
pfam_n_genera = (pfam_prev.groupby('pfam_id')['genus_lower'].nunique())
pfam_keep = pfam_n_genera[pfam_n_genera >= PFAM_MIN_GENERA].index
pfam_wide = pfam_wide_full[pfam_wide_full.columns.intersection(pfam_keep)]
print(f'Pfam domains after genus filter (>={PFAM_MIN_GENERA}): {pfam_wide.shape[1]:,}')

# Genus RA wide (samples × genera)
ra_wide_pfam = gc2.pivot_table(
    index='sample_id', columns='genus_lower', values='genus_ra', fill_value=0.0)

shared_genera_pfam = ra_wide_pfam.columns.intersection(pfam_wide.index)
ra_mat_pfam   = ra_wide_pfam[shared_genera_pfam].values.astype(np.float64)
prev_mat_pfam = pfam_wide.loc[shared_genera_pfam].values.astype(np.float64)
cwm_pfam_mat  = ra_mat_pfam @ prev_mat_pfam  # (n_samples, n_pfams)

cwm_pfam = pd.DataFrame(cwm_pfam_mat, index=ra_wide_pfam.index, columns=pfam_wide.columns)
print(f'Pfam CWM before sample filter: {cwm_pfam.shape}')

# Filter by sample coverage
pfam_sample_count = (cwm_pfam > 0).sum(axis=0)
pfam_ids = pfam_sample_count[pfam_sample_count >= PFAM_MIN_SAMPLES].index
cwm_pfam = cwm_pfam[pfam_ids]
print(f'Pfam CWM after sample filter (>={PFAM_MIN_SAMPLES}): {cwm_pfam.shape}')

cwm_pfam_aligned = cwm_pfam.reindex(base.index).fillna(0.0).values.astype(np.float64)
pfam_ids_list = list(pfam_ids)
print(f'Pfam CWM aligned: {cwm_pfam_aligned.shape}')


Pfam domains after top-5000 prevalence filter: 5,000


Pfam domains after genus filter (>=5): 5,000


Pfam CWM before sample filter: (4879, 5000)
Pfam CWM after sample filter (>=30): (4879, 5000)
Pfam CWM aligned: (4884, 5000)


In [7]:
# Compute COG CWM: same approach as Pfam CWM.
# Only 25 COG categories — no filtering needed.

cog_wide = cog_prev.pivot_table(
    index='genus_lower', columns='cog_letter', values='prevalence', fill_value=0.0)

ra_wide_cog = gc2.pivot_table(
    index='sample_id', columns='genus_lower', values='genus_ra', fill_value=0.0)

shared_genera_cog = ra_wide_cog.columns.intersection(cog_wide.index)
ra_mat_cog   = ra_wide_cog[shared_genera_cog].values.astype(np.float64)
prev_mat_cog = cog_wide.loc[shared_genera_cog].values.astype(np.float64)
cwm_cog_mat  = ra_mat_cog @ prev_mat_cog

cwm_cog = pd.DataFrame(cwm_cog_mat, index=ra_wide_cog.index, columns=cog_wide.columns)
cwm_cog_aligned = cwm_cog.reindex(base.index).fillna(0.0).values.astype(np.float64)
cog_ids_list = list(cog_wide.columns)
print(f'COG CWM: {cwm_cog.shape} (all {len(cog_ids_list)} COG categories)')
print(f'COG CWM aligned: {cwm_cog_aligned.shape}')
print(f'COG categories: {sorted(cog_ids_list)}')


COG CWM: (4879, 24) (all 24 COG categories)
COG CWM aligned: (4884, 24)
COG categories: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'S', 'T', 'U', 'V', 'W', 'Y', 'Z']


In [8]:
# Build METAL_LIST: same threshold as NB02 (>=30 positive values across all regions).
combined_idx = combined.set_index('sample_id')
metal_cols = [c for c in combined.columns if c not in ('sample_id', 'region')]

METAL_LIST = []
for m in metal_cols:
    vals = pd.to_numeric(combined_idx.reindex(base.index)[m], errors='coerce')
    n_pos = (vals > 0).sum()
    if n_pos >= 30:
        METAL_LIST.append(m)

print(f'METAL_LIST: {len(METAL_LIST)} metals with >=30 positive values')
print(METAL_LIST)


METAL_LIST: 23 metals with >=30 positive values
['As', 'B', 'Ba', 'Be', 'Cd', 'Ce', 'Co', 'Cr', 'Cu', 'Ga', 'La', 'Mo', 'Nb', 'Nd', 'Ni', 'Pb', 'Sc', 'Sr', 'V', 'Y', 'Yb', 'Zn', 'Zr']


In [9]:
# Vectorized FWL engine (shared by Pfam and COG runs).

def fwl_all_features(X, Y, Z):
    """Vectorized FWL: beta/SE/t for each column of Y regressed on X after partialling Z."""
    n, p = Z.shape
    coef_x, *_ = np.linalg.lstsq(Z, X, rcond=None)
    Mx = X - Z @ coef_x
    coef_y, *_ = np.linalg.lstsq(Z, Y, rcond=None)
    My = Y - Z @ coef_y
    MxMx = Mx @ Mx
    betas = My.T @ Mx / MxMx
    resid = My - np.outer(Mx, betas)
    dof = max(n - p - 1, 1)
    sigma2 = (resid ** 2).sum(axis=0) / dof
    se = np.sqrt(sigma2 / MxMx)
    t_stat = betas / np.where(se > 0, se, np.nan)
    return betas, se, t_stat


def run_fwl_for_annot(annot_matrix, annot_ids, label):
    """Run L0-L6 FWL for all metals x all annotation features.
    Returns long-format DataFrame: metal, level, feat_id, beta, se, t, p, n, beta_per_iqr."""
    all_results = []
    for metal in METAL_LIST:
        metal_arr = pd.to_numeric(combined_idx.reindex(base.index)[metal], errors='coerce').values
        metal_arr = np.where(metal_arr > 0, metal_arr, np.nan)
        log_metal = np.log10(metal_arr)
        iqr_log = np.nanpercentile(log_metal, 75) - np.nanpercentile(log_metal, 25)

        for level in range(7):
            Z = build_Z(level)
            valid = np.isfinite(log_metal) & np.all(np.isfinite(Z), axis=1)
            n_valid = int(valid.sum())
            if n_valid < 30:
                continue

            X = log_metal[valid]
            Y = annot_matrix[valid]
            Zs = Z[valid]

            betas, se, t_stat = fwl_all_features(X, Y, Zs)
            dof = max(n_valid - Zs.shape[1] - 1, 1)
            pvals = 2 * t_dist.sf(np.abs(t_stat), df=dof)

            df_res = pd.DataFrame({
                'metal':        metal,
                'level':        f'L{level}',
                'feat_id':      annot_ids,
                'beta':         betas,
                'se':           se,
                't_stat':       t_stat,
                'p':            pvals,
                'n':            n_valid,
                'beta_per_iqr': betas * iqr_log,
            })
            all_results.append(df_res)

        print(f'  {metal} done', end='\r')

    print(f'\n{label} FWL complete: {len(all_results)} metal×level combinations')
    return pd.concat(all_results, ignore_index=True)


In [10]:
# Run FWL for Pfam CWM × metals.
print(f'Running Pfam FWL: {len(METAL_LIST)} metals × 7 levels × {cwm_pfam_aligned.shape[1]:,} Pfams')
results_pfam = run_fwl_for_annot(cwm_pfam_aligned, pfam_ids_list, 'Pfam')

# BH-FDR within each metal × level
fdr_pfam = []
for (metal, level), grp in results_pfam.groupby(['metal', 'level']):
    valid_p = grp['p'].notna() & grp['p'].between(0, 1)
    q = np.full(len(grp), np.nan)
    if valid_p.sum() > 0:
        _, q_vals, _, _ = multipletests(grp.loc[valid_p, 'p'], method='fdr_bh')
        q[valid_p.values] = q_vals
    grp = grp.copy()
    grp['q_bh'] = q
    fdr_pfam.append(grp)

fdr_pfam = pd.concat(fdr_pfam, ignore_index=True)
fdr_pfam.attrs = {}
fdr_pfam.to_parquet(DATA / 'nb08_pfam_fwl_fdr.parquet', index=False)

pfam_L1_hits = fdr_pfam[(fdr_pfam['level'] == 'L1') & (fdr_pfam['q_bh'] < 0.05)]
print('Pfam FDR hits at L1 (pH-adjusted), by metal:')
print(pfam_L1_hits.groupby('metal')['feat_id'].count().to_string())
print(f'\nTotal Pfam L1 FDR hits: {len(pfam_L1_hits):,}')


Running Pfam FWL: 23 metals × 7 levels × 5,000 Pfams


  Zr done
Pfam FWL complete: 161 metal×level combinations


Pfam FDR hits at L1 (pH-adjusted), by metal:
metal
Cd    761

Total Pfam L1 FDR hits: 761


In [11]:
# Run FWL for COG CWM × metals.
print(f'Running COG FWL: {len(METAL_LIST)} metals × 7 levels × {cwm_cog_aligned.shape[1]} COG categories')
results_cog = run_fwl_for_annot(cwm_cog_aligned, cog_ids_list, 'COG')

# BH-FDR
fdr_cog = []
for (metal, level), grp in results_cog.groupby(['metal', 'level']):
    valid_p = grp['p'].notna() & grp['p'].between(0, 1)
    q = np.full(len(grp), np.nan)
    if valid_p.sum() > 0:
        _, q_vals, _, _ = multipletests(grp.loc[valid_p, 'p'], method='fdr_bh')
        q[valid_p.values] = q_vals
    grp = grp.copy()
    grp['q_bh'] = q
    fdr_cog.append(grp)

fdr_cog = pd.concat(fdr_cog, ignore_index=True)
fdr_cog.attrs = {}
fdr_cog.to_parquet(DATA / 'nb08_cog_fwl_fdr.parquet', index=False)

cog_L1_hits = fdr_cog[(fdr_cog['level'] == 'L1') & (fdr_cog['q_bh'] < 0.05)]
print('COG FDR hits at L1, by metal:')
print(cog_L1_hits.groupby('metal')['feat_id'].count().to_string())
print(f'\nTotal COG L1 FDR hits: {len(cog_L1_hits):,}')
if len(cog_L1_hits) > 0:
    print('\nCOG categories significant at L1:')
    print(cog_L1_hits[['metal','feat_id','beta','q_bh']].sort_values('q_bh').to_string(index=False))


Running COG FWL: 23 metals × 7 levels × 24 COG categories


  Zr done
COG FWL complete: 161 metal×level combinations
COG FDR hits at L1, by metal:
metal
Cd     2
Cu     1
Pb     1
V     18

Total COG L1 FDR hits: 22

COG categories significant at L1:
metal feat_id      beta     q_bh
   Cd       Z  0.067232 0.004838
   Cd       W  0.039922 0.024828
   Cu       Z  0.041155 0.032188
    V       C -0.056001 0.039585
    V       E -0.056001 0.039585
    V       D -0.056001 0.039585
    V       F -0.056001 0.039585
    V       G -0.056001 0.039585
    V       K -0.056001 0.039585
    V       H -0.056001 0.039585
    V       I -0.055998 0.039585
    V       J -0.056001 0.039585
    V       M -0.056001 0.039585
    V       L -0.056001 0.039585
    V       O -0.056001 0.039585
    V       P -0.056001 0.039585
    V       U -0.056001 0.039585
    V       Q -0.055966 0.039585
    V       S -0.056001 0.039585
    V       T -0.056001 0.039585
    V       V -0.055995 0.039585
   Pb       Z  0.050242 0.042698


In [12]:
# Compare KO vs Pfam vs COG: hit counts per metal at each level.
ko_fdr = pd.read_parquet(DATA / 'nb02_fwl_results_fdr.parquet')
ko_fdr = ko_fdr.rename(columns={'ko_id': 'feat_id'})

summary_rows = []
for level in ['L0', 'L1', 'L2', 'L3', 'L4', 'L5', 'L6']:
    for metal in METAL_LIST:
        n_ko   = ((ko_fdr['level']==level) & (ko_fdr['metal']==metal) & (ko_fdr['q_bh']<0.05)).sum()
        n_pfam = ((fdr_pfam['level']==level) & (fdr_pfam['metal']==metal) & (fdr_pfam['q_bh']<0.05)).sum()
        n_cog  = ((fdr_cog['level']==level) & (fdr_cog['metal']==metal) & (fdr_cog['q_bh']<0.05)).sum()
        summary_rows.append({'level': level, 'metal': metal,
                              'n_ko': n_ko, 'n_pfam': n_pfam, 'n_cog': n_cog})

summary = pd.DataFrame(summary_rows)
summary.attrs = {}
summary.to_csv(DATA / 'nb08_ko_pfam_cog_comparison.csv', index=False)

print('=== L1 FDR hit counts by annotation type ===')
L1_summary = summary[summary['level'] == 'L1'].set_index('metal')[['n_ko','n_pfam','n_cog']]
print(L1_summary.to_string())
print(f'\nL1 totals: KO={L1_summary["n_ko"].sum():,}  Pfam={L1_summary["n_pfam"].sum():,}  COG={L1_summary["n_cog"].sum():,}')

# Direction consistency at L1 (for metals with both KO and Pfam hits)
# Group KO hits by metal and report mean beta sign; compare with Pfam hits
ko_L1 = ko_fdr[(ko_fdr['level']=='L1') & (ko_fdr['q_bh']<0.05)]
pfam_L1_hits = fdr_pfam[(fdr_pfam['level']=='L1') & (fdr_pfam['q_bh']<0.05)]
print('\n=== Mean beta sign at L1: KO vs Pfam ===')
for metal in METAL_LIST:
    ko_b   = ko_L1[ko_L1['metal']==metal]['beta'].mean()
    pfam_b = pfam_L1_hits[pfam_L1_hits['metal']==metal]['beta'].mean()
    cog_b  = cog_L1_hits[cog_L1_hits['metal']==metal]['beta'].mean() if len(cog_L1_hits[cog_L1_hits['metal']==metal]) else float('nan')
    n_ko   = len(ko_L1[ko_L1['metal']==metal])
    n_pfam = len(pfam_L1_hits[pfam_L1_hits['metal']==metal])
    if n_ko > 0 or n_pfam > 0:
        print(f'  {metal:3s}  KO β={ko_b:+.4f} (n={n_ko})  Pfam β={pfam_b:+.4f} (n={n_pfam})  COG β={cog_b:+.4f}')


=== L1 FDR hit counts by annotation type ===
       n_ko  n_pfam  n_cog
metal                     
As       41       0      0
B         0       0      0
Ba        0       0      0
Be        0       0      0
Cd       10     761      2
Ce        0       0      0
Co        0       0      0
Cr       38       0      0
Cu        0       0      1
Ga        0       0      0
La        9       0      0
Mo        0       0      0
Nb        1       0      0
Nd      235       0      0
Ni       38       0      0
Pb       14       0      1
Sc        0       0      0
Sr        1       0      0
V         6       0     18
Y         0       0      0
Yb       91       0      0
Zn       76       0      0
Zr        0       0      0

L1 totals: KO=560  Pfam=761  COG=22

=== Mean beta sign at L1: KO vs Pfam ===
  As   KO β=-0.0037 (n=41)  Pfam β=+nan (n=0)  COG β=+nan
  Cd   KO β=+0.0280 (n=10)  Pfam β=+0.0474 (n=761)  COG β=+0.0536
  Cr   KO β=-0.0029 (n=38)  Pfam β=+nan (n=0)  COG β=+nan
  La   KO β=-0.0284

In [13]:
# Figure 1: Hit counts by annotation type across levels.
summary_agg = summary.groupby('level')[['n_ko','n_pfam','n_cog']].sum().loc[
    ['L0','L1','L2','L3','L4','L5','L6']]

fig, ax = plt.subplots(figsize=(FIGW['1.5col'], ROW_H))
x = np.arange(len(summary_agg))
w = 0.25
ax.bar(x - w, summary_agg['n_ko'],   width=w, label='KO',   color=PALETTE[0], edgecolor='k', linewidth=0.5)
ax.bar(x,     summary_agg['n_pfam'], width=w, label='Pfam', color=PALETTE[1], edgecolor='k', linewidth=0.5)
ax.bar(x + w, summary_agg['n_cog'],  width=w, label='COG',  color=PALETTE[2], edgecolor='k', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(['L0','L1','L2','L3','L4','L5','L6'])
ax.set_xlabel('Causal level')
ax.set_ylabel('Total FDR hits (q < 0.05)')
ax.set_title('KO vs Pfam vs COG: FDR hit counts by level', fontsize=10)
ax.legend(fontsize=8)
grid_h(ax)
save(fig, FIGS / 'fig_nb08_hits_by_level')

# Figure 2: L1 hits per metal, stacked by annotation type.
fig2, ax2 = plt.subplots(figsize=(FIGW['2col'], ROW_H))
metals_sorted = sorted(METAL_LIST)
L1_s = L1_summary.reindex(metals_sorted).fillna(0)
x2 = np.arange(len(metals_sorted))
ax2.bar(x2 - w, L1_s['n_ko'],   width=w, label='KO',   color=PALETTE[0], edgecolor='k', linewidth=0.5)
ax2.bar(x2,     L1_s['n_pfam'], width=w, label='Pfam', color=PALETTE[1], edgecolor='k', linewidth=0.5)
ax2.bar(x2 + w, L1_s['n_cog'],  width=w, label='COG',  color=PALETTE[2], edgecolor='k', linewidth=0.5)
ax2.set_xticks(x2)
ax2.set_xticklabels(metals_sorted, rotation=45, ha='right', fontsize=8)
ax2.set_xlabel('Metal')
ax2.set_ylabel('L1 FDR hits')
ax2.set_title('L1 FDR hits by metal and annotation type', fontsize=10)
ax2.legend(fontsize=8)
grid_h(ax2)
save(fig2, FIGS / 'fig_nb08_L1_hits_by_metal')

# Figure 3: Top COG categories at L1 (all metals).
if len(cog_L1_hits) > 0:
    COG_NAMES = {
        'J': 'Translation', 'A': 'RNA processing', 'K': 'Transcription',
        'L': 'DNA replication/repair', 'B': 'Chromatin', 'D': 'Cell division',
        'Y': 'Nuclear structure', 'V': 'Defense', 'T': 'Signal transduction',
        'M': 'Cell wall/membrane', 'N': 'Cell motility', 'Z': 'Cytoskeleton',
        'W': 'Extracellular', 'U': 'Intracellular trafficking',
        'O': 'Posttranslational modification', 'X': 'Mobilome',
        'C': 'Energy production', 'G': 'Carbohydrate metabolism',
        'E': 'Amino acid metabolism', 'F': 'Nucleotide metabolism',
        'H': 'Coenzyme metabolism', 'I': 'Lipid metabolism',
        'P': 'Inorganic ion transport', 'Q': 'Secondary metabolites',
        'R': 'General function', 'S': 'Unknown',
    }
    cog_pivot = cog_L1_hits.groupby('feat_id')['metal'].count().sort_values(ascending=False)
    fig3, ax3 = plt.subplots(figsize=(FIGW['1.5col'], ROW_H))
    labels = [f"{c} ({COG_NAMES.get(c,'?')[:18]})" for c in cog_pivot.index]
    ax3.barh(labels, cog_pivot.values, color=PALETTE[2], edgecolor='k', linewidth=0.5)
    ax3.set_xlabel('Number of metals with FDR hit')
    ax3.set_ylabel('COG category')
    ax3.set_title('COG categories significant at L1', fontsize=10)
    grid_h(ax3)
    save(fig3, FIGS / 'fig_nb08_cog_L1_categories')

# Figure 4: Stability of Pfam L1 hits across levels (for top metals).
top_metal = L1_summary['n_pfam'].idxmax() if L1_summary['n_pfam'].max() > 0 else None
if top_metal is not None:
    pfam_top = fdr_pfam[(fdr_pfam['metal'] == top_metal) & (fdr_pfam['q_bh'] < 0.05) &
                         (fdr_pfam['level'] == 'L1')]['feat_id'].tolist()
    if pfam_top:
        levels_ord = ['L0','L1','L2','L3','L4','L5','L6']
        pfam_stability = fdr_pfam[
            (fdr_pfam['metal'] == top_metal) & (fdr_pfam['feat_id'].isin(pfam_top[:20]))
        ].pivot_table(index='feat_id', columns='level', values='beta')[levels_ord]

        fig4, ax4 = plt.subplots(figsize=(FIGW['1.5col'], ROW_H))
        for i, fid in enumerate(pfam_stability.index[:10]):
            ax4.plot(levels_ord, pfam_stability.loc[fid], color=PALETTE[i % len(PALETTE)],
                     lw=1, alpha=0.8, label=fid[:15])
        ax4.axhline(0, color='gray', lw=0.8, ls='--')
        ax4.set_xlabel('Causal level')
        ax4.set_ylabel('β (Pfam CWM ~ log10 metal)')
        ax4.set_title(f'Pfam L1 hits stability across levels ({top_metal})', fontsize=10)
        ax4.legend(fontsize=7, ncol=2)
        grid_h(ax4)
        save(fig4, FIGS / f'fig_nb08_pfam_stability_{top_metal}')

print('Figures saved.')


Figures saved.


In [14]:
# Compact summary for REPORT.md
print('=== NB08 Summary ===')
print(f'Pfam domains tested: {len(pfam_ids_list):,}')
print(f'COG categories tested: {len(cog_ids_list)}')
print()
print('L1 FDR hits (q<0.05):')
for metal in METAL_LIST:
    n_ko   = ((ko_fdr['level']=='L1') & (ko_fdr['metal']==metal) & (ko_fdr['q_bh']<0.05)).sum()
    n_pfam = ((fdr_pfam['level']=='L1') & (fdr_pfam['metal']==metal) & (fdr_pfam['q_bh']<0.05)).sum()
    n_cog  = ((fdr_cog['level']=='L1') & (fdr_cog['metal']==metal) & (fdr_cog['q_bh']<0.05)).sum()
    if n_ko > 0 or n_pfam > 0 or n_cog > 0:
        print(f'  {metal:3s}: KO={n_ko:4d}  Pfam={n_pfam:4d}  COG={n_cog:2d}')

print()
print('Pfam L1 top hits (by q):')
if len(pfam_L1_hits) > 0:
    top10 = pfam_L1_hits.sort_values('q_bh').head(10)
    print(top10[['metal','feat_id','n','beta','q_bh']].to_string(index=False))

print()
print('COG L1 hits:')
if len(cog_L1_hits) > 0:
    print(cog_L1_hits[['metal','feat_id','beta','q_bh']].sort_values(['metal','q_bh']).to_string(index=False))
else:
    print('  None')


=== NB08 Summary ===
Pfam domains tested: 5,000
COG categories tested: 24

L1 FDR hits (q<0.05):
  As : KO=  41  Pfam=   0  COG= 0
  Cd : KO=  10  Pfam= 761  COG= 2
  Cr : KO=  38  Pfam=   0  COG= 0
  Cu : KO=   0  Pfam=   0  COG= 1
  La : KO=   9  Pfam=   0  COG= 0


  Nb : KO=   1  Pfam=   0  COG= 0
  Nd : KO= 235  Pfam=   0  COG= 0
  Ni : KO=  38  Pfam=   0  COG= 0
  Pb : KO=  14  Pfam=   0  COG= 1
  Sr : KO=   1  Pfam=   0  COG= 0
  V  : KO=   6  Pfam=   0  COG=18
  Yb : KO=  91  Pfam=   0  COG= 0
  Zn : KO=  76  Pfam=   0  COG= 0

Pfam L1 top hits (by q):
metal         feat_id   n     beta     q_bh
   Cd      Beta_helix 998 0.086224 0.000551
   Cd             DHC 998 0.021314 0.002729
   Cd  Zn_peptidase_2 998 0.062143 0.004308
   Cd         DUF5122 998 0.036153 0.004308
   Cd         DUF2851 998 0.037246 0.004308
   Cd   N-glycanase_C 998 0.013489 0.004508
   Cd  Glyco_transf_5 998 0.083895 0.004508
   Cd Thymidylat_synt 998 0.095222 0.004508
   Cd Cleaved_Adhesin 998 0.014045 0.004508
   Cd       SprT-like 998 0.070959 0.004536

COG L1 hits:
metal feat_id      beta     q_bh
   Cd       Z  0.067232 0.004838
   Cd       W  0.039922 0.024828
   Cu       Z  0.041155 0.032188
   Pb       Z  0.050242 0.042698
    V       C -0.056001 0.039585
    V 

## Findings

*(Populate after execution — see summary cell output above)*

### Pfam CWM × metals (L1)

- N Pfam domains tested: (fill from summary)
- L1 FDR hits per metal: (fill from summary)

### COG CWM × metals (L1)

- N COG categories tested: (fill — all 25 or subset)
- Significant COG categories: (fill from summary)

### KO / Pfam / COG comparison

- Hit count ratio (Pfam/KO): (fill)
- Direction consistency: (fill)
- Interpretation: (fill)
